# GeoRisk Location Intelligence System (GLIS)

###By Rajat Tiwari

In [ ]:

!pip -q install pandas numpy scikit-learn plotly

In [ ]:
#Imports
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
FILE_PATH = "/content/NRI_Table_Counties.csv"
df = pd.read_csv(FILE_PATH, low_memory=False)

print("Original shape:", df.shape)
print("\nFirst 10 columns:")
print(df.columns[:10].tolist())

# Standardize columns
df.columns = [c.strip().upper() for c in df.columns]

print("\nStandardized columns sample:")
print(df.columns[:20].tolist())

Original shape: (3232, 465)

First 10 columns:
['OID_', 'NRI_ID', 'STATE', 'STATEABBRV', 'STATEFIPS', 'COUNTY', 'COUNTYTYPE', 'COUNTYFIPS', 'STCOFIPS', 'POPULATION']

Standardized columns sample:
['OID_', 'NRI_ID', 'STATE', 'STATEABBRV', 'STATEFIPS', 'COUNTY', 'COUNTYTYPE', 'COUNTYFIPS', 'STCOFIPS', 'POPULATION', 'BUILDVALUE', 'AGRIVALUE', 'AREA', 'RISK_VALUE', 'RISK_SCORE', 'RISK_RATNG', 'RISK_SPCTL', 'EAL_SCORE', 'EAL_RATNG', 'EAL_SPCTL']


In [ ]:
TARGET_LOCATIONS = {
    "Honolulu, Hawaii": "15003",
    "Los Angeles, California": "06037",
    "Miami, Florida": "12086",
    "San Francisco, California": "06075",
    "New York, New York": "36061"
}

In [ ]:
#Find FIPS column
possible_fips_cols = ["STCOFIPS", "COUNTYFIPS", "FIPS", "GEOID"]
fips_col = None

for c in possible_fips_cols:
    if c in df.columns:
        fips_col = c
        break

if fips_col is None:
    raise ValueError(f"Could not find a county FIPS column. Checked: {possible_fips_cols}")

df["FIPS_CODE"] = df[fips_col].astype(str).str.replace(".0", "", regex=False).str.zfill(5)

In [ ]:
#Build display name
county_col = "COUNTY" if "COUNTY" in df.columns else None
state_col = "STATE" if "STATE" in df.columns else None

if county_col and state_col:
    df["DISPLAY_NAME"] = df[county_col].astype(str) + ", " + df[state_col].astype(str)
elif county_col:
    df["DISPLAY_NAME"] = df[county_col].astype(str)
else:
    df["DISPLAY_NAME"] = df["FIPS_CODE"]

In [ ]:
#Detect usable FEMA columns automatically

alias_map = {
    "risk": ["RISK_SCORE", "WNTW_RISKS", "RISKS", "WNTW_RISKV"],
    "eal": ["EAL_SCORE", "WNTW_EALS", "EALS"],
    "sovi": ["SOVI_SCORE", "SOVI", "SOCIAL_VULNERABILITY_SCORE"],
    "resl": ["RESL_SCORE", "RESL", "COMMUNITY_RESILIENCE_SCORE", "RESILIENCE_SCORE"]
}

selected = {}

for key, aliases in alias_map.items():
    for col in aliases:
        if col in df.columns:
            selected[key] = col
            break

print("\nDetected columns:")
print(selected)

# We need at least risk and eal to proceed safely
required_min = ["risk", "eal"]
missing_min = [k for k in required_min if k not in selected]
if missing_min:
    raise ValueError(f"Required columns not found for: {missing_min}")

# Convert selected columns to numeric
for logical_name, col in selected.items():
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Keep only rows with the required minimum columns present
needed_cols = [selected[k] for k in selected]
df = df.dropna(subset=[selected["risk"], selected["eal"]]).copy()

print("\nShape after cleaning:", df.shape)


Detected columns:
{'risk': 'RISK_SCORE', 'eal': 'EAL_SCORE', 'sovi': 'SOVI_SCORE', 'resl': 'RESL_SCORE'}

Shape after cleaning: (3144, 467)


In [ ]:
#Feature prep
# Higher transformed value = safer
model_features = pd.DataFrame(index=df.index)

# Lower risk is better
model_features["SAFE_RISK"] = -df[selected["risk"]]

# Lower expected annual loss is better
model_features["SAFE_EAL"] = -df[selected["eal"]]

# Lower social vulnerability is better, if available
if "sovi" in selected:
    model_features["SAFE_SOVI"] = -df[selected["sovi"]]

# Higher resilience is better, if available
if "resl" in selected:
    model_features["SAFE_RESL"] = df[selected["resl"]]

# Drop rows with NA in any chosen feature
valid_index = model_features.dropna().index
df = df.loc[valid_index].copy()
model_features = model_features.loc[valid_index].copy()

print("\nModel features used:")
print(model_features.columns.tolist())


Model features used:
['SAFE_RISK', 'SAFE_EAL', 'SAFE_SOVI', 'SAFE_RESL']


In [ ]:
#Scale features

scaler = StandardScaler()
X = scaler.fit_transform(model_features)

#Train unsupervised model
N_CLUSTERS = 4
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(X)
sil = silhouette_score(X, cluster_labels)

df["CLUSTER"] = cluster_labels

centroids = pd.DataFrame(kmeans.cluster_centers_, columns=model_features.columns)
centroids["SAFETY_CENTROID_SCORE"] = centroids.sum(axis=1)

safest_cluster = int(centroids["SAFETY_CENTROID_SCORE"].idxmax())
safe_center = kmeans.cluster_centers_[safest_cluster]

# Distance to safest centroid
distances = np.linalg.norm(X - safe_center, axis=1)

# Convert distance to safety score (0–100)
max_d = distances.max()
min_d = distances.min()

if max_d == min_d:
    df["SAFETY_SCORE"] = 100.0
else:
    df["SAFETY_SCORE"] = 100 * (max_d - distances) / (max_d - min_d)

df["IS_SAFEST_CLUSTER"] = df["CLUSTER"] == safest_cluster

print(f"\nSilhouette Score: {sil:.4f}")
print(f"Safest Cluster ID: {safest_cluster}")


Silhouette Score: 0.3213
Safest Cluster ID: 2


In [ ]:
#Filter your target locations

candidate_df = df[df["FIPS_CODE"].isin(TARGET_LOCATIONS.values())].copy()

reverse_map = {v: k for k, v in TARGET_LOCATIONS.items()}
candidate_df["LOCATION"] = candidate_df["FIPS_CODE"].map(reverse_map)

if candidate_df.empty:
    raise ValueError("None of the target FIPS codes were found in the file.")

candidate_ranked = candidate_df.sort_values("SAFETY_SCORE", ascending=False).reset_index(drop=True)
candidate_ranked.index = candidate_ranked.index + 1
candidate_ranked.index.name = "RANK"


#Output table

result_cols = ["LOCATION", "FIPS_CODE", "CLUSTER", "SAFETY_SCORE"]

for key in ["risk", "eal", "sovi", "resl"]:
    if key in selected:
        result_cols.append(selected[key])

print("\n=== Ranked Candidate Locations ===")
print(candidate_ranked[result_cols].to_string())

best_location = candidate_ranked.iloc[0]["LOCATION"]
best_score = candidate_ranked.iloc[0]["SAFETY_SCORE"]

print(f"\nBest first location: {best_location}")
print(f"Safety score: {best_score:.2f}/100")


=== Ranked Candidate Locations ===
                       LOCATION FIPS_CODE  CLUSTER  SAFETY_SCORE  RISK_SCORE   EAL_SCORE  SOVI_SCORE  RESL_SCORE
RANK                                                                                                            
1              Honolulu, Hawaii     15003        0     25.657999   98.759542   99.040842   25.890585   62.627226
2     San Francisco, California     06075        0     18.379925   99.522901   99.597772   48.059796   99.650127
3                Miami, Florida     12086        3     15.290924   99.618321   99.566832   75.858779   42.907125
4       Los Angeles, California     06037        3     13.865197  100.000000  100.000000   55.375318   13.549618
5            New York, New York     36061        3      9.466547   98.791349   97.834158   96.310433   67.366412

Best first location: Honolulu, Hawaii
Safety score: 25.66/100


In [ ]:
#Bar chart
fig_bar = px.bar(
    candidate_ranked,
    x="LOCATION",
    y="SAFETY_SCORE",
    color="SAFETY_SCORE",
    text="SAFETY_SCORE",
    title="Candidate Location Safety Scores"
)

fig_bar.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig_bar.update_layout(
    template="plotly_dark",
    height=550,
    xaxis_title="Location",
    yaxis_title="Safety Score (0–100)",
    coloraxis_showscale=False,
    paper_bgcolor="#0b1020",
    plot_bgcolor="#0b1020",
    font=dict(size=14)
)
fig_bar.show()

Insights:

Our bar chart compares the calculated safety scores of the selected cities based on FEMA disaster risk indicators.

From the results, Honolulu, Hawaii has the highest safety score, indicating relatively lower disaster risk and better resilience compared to the other cities.

Cities like San Francisco and Miami fall in the middle range, while Los Angeles and New York show lower safety scores, suggesting relatively higher exposure to disaster risk factors.

This visualization helps quickly identify which locations may be safer for infrastructure investment or development.

In [ ]:
#Radar chart
radar_df = candidate_ranked.copy()

radar_components = {}

radar_components["Lower Risk"] = -radar_df[selected["risk"]]
radar_components["Lower EAL"] = -radar_df[selected["eal"]]

if "sovi" in selected:
    radar_components["Lower Social Vulnerability"] = -radar_df[selected["sovi"]]

if "resl" in selected:
    radar_components["Higher Resilience"] = radar_df[selected["resl"]]

# Normalize 0-100 inside candidate set for display
radar_plot_df = pd.DataFrame(index=radar_df.index)

for label, values in radar_components.items():
    mn, mx = values.min(), values.max()
    if mx == mn:
        radar_plot_df[label] = 100
    else:
        radar_plot_df[label] = 100 * (values - mn) / (mx - mn)

radar_plot_df["LOCATION"] = radar_df["LOCATION"].values

theta_labels = list(radar_components.keys())

fig_radar = go.Figure()

for _, row in radar_plot_df.iterrows():
    r_vals = [row[label] for label in theta_labels]
    r_vals.append(r_vals[0])

    theta_vals = theta_labels + [theta_labels[0]]

    fig_radar.add_trace(go.Scatterpolar(
        r=r_vals,
        theta=theta_vals,
        fill="toself",
        name=row["LOCATION"]
    ))

fig_radar.update_layout(
    template="plotly_dark",
    title="Risk / Resilience Profile Comparison",
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    height=700,
    paper_bgcolor="#0b1020",
    plot_bgcolor="#0b1020",
    font=dict(size=14)
)
fig_radar.show()

Insights:

The radar chart compares cities across multiple dimensions including disaster risk, expected annual loss, social vulnerability, and resilience.

From the visualization we observe that:
Honolulu performs well in lower risk and lower social vulnerability.

San Francisco shows strong resilience compared to other cities.

Miami and Los Angeles show moderate risk profiles.

New York shows higher vulnerability across some risk indicators.

This chart highlights that each city has different strengths and weaknesses across risk factors, which is important for comprehensive geographic decision-making.

In [ ]:
#Scatter plot of county clusters
plot_df = pd.DataFrame(X[:, :2], columns=["DIM1", "DIM2"])
plot_df["CLUSTER"] = df["CLUSTER"].values
plot_df["FIPS_CODE"] = df["FIPS_CODE"].values
plot_df["DISPLAY_NAME"] = df["DISPLAY_NAME"].values
plot_df["IS_TARGET"] = plot_df["FIPS_CODE"].isin(TARGET_LOCATIONS.values())
plot_df["TARGET_LABEL"] = plot_df["FIPS_CODE"].map(reverse_map)

fig_scatter = px.scatter(
    plot_df,
    x="DIM1",
    y="DIM2",
    color=plot_df["CLUSTER"].astype(str),
    hover_data=["DISPLAY_NAME", "FIPS_CODE"],
    title="County Clusters (first two standardized dimensions)",
    opacity=0.35
)

target_points = plot_df[plot_df["IS_TARGET"]].copy()

fig_scatter.add_trace(go.Scatter(
    x=target_points["DIM1"],
    y=target_points["DIM2"],
    mode="markers+text",
    text=target_points["TARGET_LABEL"],
    textposition="top center",
    marker=dict(size=14, symbol="diamond"),
    name="Target Locations"
))

fig_scatter.update_layout(
    template="plotly_dark",
    height=650,
    paper_bgcolor="#0b1020",
    plot_bgcolor="#0b1020",
    font=dict(size=14)
)
fig_scatter.show()

#Save output
candidate_ranked[result_cols].to_csv("candidate_safety_ranking.csv", index=True)
print("\nSaved file: candidate_safety_ranking.csv")


Saved file: candidate_safety_ranking.csv


Insights:

The scatter plot shows clusters of counties grouped based on similar disaster risk characteristics using machine learning clustering.

Key observations include:
Counties naturally form distinct clusters representing different levels of disaster risk.

Target cities appear within specific clusters, indicating their relative risk category compared to other counties in the United States.

Counties closer to the safer cluster represent regions with lower disaster risk and higher resilience.

This visualization confirms that the clustering model successfully identifies patterns in geographic disaster risk across counties.